# 07 Plain softmax against CassaCheck at matched coverage

The baseline is the same frozen network with a plain softmax: no temperature. It gets its own abstention threshold, fitted exactly like CassaCheck's: the 20th percentile of top confidence on the validation fit half (target coverage 0.80). Both thresholds are fixed in cell 4, before any test logit is loaded.

Compared on the internal test and on CCMT: wrong answers given, coverage, accuracy on answered photos, wrong answers at exactly the same number of answered photos, wrong answers shown with 90% confidence or more, ECE, and the area under the risk-coverage curve. Paired bootstrap, 2,000 resamples over images.

Test-set note: notebook 04b did not save the test logits, so cell 6 recomputes them with the same frozen checkpoints and the same preprocessing, checks that accuracy and coverage reproduce notebook 04b exactly, and saves them so they are never recomputed again. Nothing is fitted on the test set or on CCMT.

In [ ]:
# Cell 1: environment, Drive, Kaggle (same as notebook 04b)
from google.colab import drive
drive.mount("/content/drive")
!pip install --quiet --upgrade kaggle scikit-learn
import os, json, torch, numpy as np, pandas as pd, torch.nn as nn
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")  # Colab Secrets, never in the notebook
!mkdir --parents ~/.kaggle && echo "$KAGGLE_API_TOKEN" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
RUNS_DIRECTORY = "/content/drive/MyDrive/cassacheck_runs"
RESULTS_DIRECTORY = "/content/drive/MyDrive/cassacheck_results"
device = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "healthy"]
print("device:", device)

In [ ]:
# Cell 2: split and validation halves, identical to notebook 04b
from sklearn.model_selection import train_test_split
!wget --quiet --output-document /content/split_v1_seed42.csv https://raw.githubusercontent.com/Justine-abc/cassavacheck/main/data/split_v1_seed42.csv
labels = pd.read_csv("/content/split_v1_seed42.csv")
validation_frame = labels[labels.split == "validation"].reset_index(drop=True)
test_frame = labels[labels.split == "test"].reset_index(drop=True)
TARGET_COVERAGE = 0.80
fit_index, measure_index = train_test_split(np.arange(len(validation_frame)), test_size=0.5, stratify=validation_frame["label"], random_state=42)
print("validation", len(validation_frame), "| fit half", len(fit_index), "| test", len(test_frame))

In [ ]:
# Cell 3: metric functions (ECE with 15 bins, as in notebooks 04b and 05)
from sklearn.metrics import f1_score

def expected_calibration_error(probabilities, targets, bins=15):
    confidence, prediction = probabilities.max(1)
    correct = (prediction == targets).float()
    edges = torch.linspace(0, 1, bins + 1); error = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (confidence > low) & (confidence <= high)
        if in_bin.any(): error += in_bin.float().mean() * (correct[in_bin].mean() - confidence[in_bin].mean()).abs()
    return float(error)

def area_under_risk_coverage(confidence, correct):
    order = np.argsort(-confidence, kind="stable")
    risk = 1 - np.cumsum(correct[order]) / np.arange(1, len(order) + 1)
    return float(risk.mean())

In [ ]:
# Cell 4: both thresholds fixed on the validation fit half, before any test logit is loaded
frozen = json.load(open(f"{RESULTS_DIRECTORY}/frozen_models.json"))
thresholds = {}
for backbone, spec in frozen.items():
    saved = torch.load(f"{RUNS_DIRECTORY}/{backbone}_seed{spec['seed']}_validation_logits.pt")
    fit_logits = saved["logits"][fit_index]
    plain = float(torch.quantile(torch.softmax(fit_logits, 1).max(1).values, 1 - TARGET_COVERAGE))
    recomputed = float(torch.quantile(torch.softmax(fit_logits / spec["temperature"], 1).max(1).values, 1 - TARGET_COVERAGE))
    assert abs(recomputed - spec["threshold"]) < 5e-4, "CassaCheck threshold does not reproduce notebook 04b"
    thresholds[backbone] = {"temperature": spec["temperature"], "cassacheck_threshold": spec["threshold"], "plain_softmax_threshold": round(plain, 4)}
    print(backbone, thresholds[backbone])
json.dump(thresholds, open(f"{RESULTS_DIRECTORY}/matched_coverage_thresholds.json", "w"), indent=2)
print("fixed; nothing below changes these values")

In [ ]:
# Cell 5: model loading and logits, identical to notebook 04b cell 6
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
!kaggle competitions download --competition cassava-leaf-disease-classification --path /content/data
!unzip -q -o /content/data/cassava-leaf-disease-classification.zip "train_images/*" -d /content/data
image_directory = "/content/data/train_images"

class CassavaDataset(Dataset):
    def __init__(self, frame, transform): self.frame, self.transform = frame, transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return self.transform(Image.open(f"{image_directory}/{row.image_id}").convert("RGB")), int(row.label)

eval_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def build_model(backbone):
    if backbone == "mobilenetv3_large":
        network = models.mobilenet_v3_large(weights=None); network.classifier[3] = nn.Linear(network.classifier[3].in_features, 5)
    else:
        network = models.resnet18(weights=None); network.fc = nn.Linear(network.fc.in_features, 5)
    return network.to(device).eval()

def logits_on(frame, backbone, checkpoint):
    network = build_model(backbone); network.load_state_dict(torch.load(f"{RUNS_DIRECTORY}/{checkpoint}", map_location=device))
    loader = DataLoader(CassavaDataset(frame, eval_transform), batch_size=128, num_workers=2)
    logits, targets = [], []
    with torch.no_grad():
        for images, batch_targets in loader:
            logits.append(network(images.to(device)).cpu()); targets.append(batch_targets)
    return torch.cat(logits), torch.cat(targets)

In [ ]:
# Cell 6: test logits of the two frozen checkpoints. Recomputed because notebook 04b did not save them; must reproduce 04b, then saved.
reference = json.load(open(f"{RESULTS_DIRECTORY}/internal_test_frozen_models.json"))
test_logits = {}
for backbone, spec in frozen.items():
    path = f"{RESULTS_DIRECTORY}/test_logits_{backbone}_seed{spec['seed']}.pt"
    if os.path.exists(path):
        saved = torch.load(path); logits, targets = saved["logits"], saved["targets"]
    else:
        logits, targets = logits_on(test_frame, backbone, spec["checkpoint"]); torch.save({"logits": logits, "targets": targets}, path)
    after = torch.softmax(logits / spec["temperature"], 1)
    accuracy = float((after.argmax(1) == targets).float().mean()); coverage = float((after.max(1).values >= spec["threshold"]).float().mean())
    print(backbone, "accuracy", round(accuracy, 4), "vs 04b", round(reference[backbone]["accuracy"], 4), "| coverage", round(coverage, 4), "vs 04b", round(reference[backbone]["coverage"], 4))
    assert abs(accuracy - reference[backbone]["accuracy"]) < 1e-3 and abs(coverage - reference[backbone]["coverage"]) < 1e-3, "does not reproduce notebook 04b"
    test_logits[backbone] = (logits, targets)

In [ ]:
# Cell 7: the comparison, with a paired bootstrap over images
rng = np.random.default_rng(42)

def compare(logits, targets, spec):
    plain = torch.softmax(logits, 1); calibrated = torch.softmax(logits / spec["temperature"], 1)
    prediction = calibrated.argmax(1).numpy()  # the temperature does not change the predicted class
    assert (plain.argmax(1).numpy() == prediction).all()
    correct = (prediction == targets.numpy()).astype(float)
    plain_confidence, calibrated_confidence = plain.max(1).values.numpy(), calibrated.max(1).values.numpy()
    plain_kept = plain_confidence >= spec["plain_softmax_threshold"]; calibrated_kept = calibrated_confidence >= spec["cassacheck_threshold"]
    # exactly matched coverage: plain softmax answers the same number of photos, its most confident ones
    k = int(calibrated_kept.sum()); matched_kept = np.zeros(len(correct), bool); matched_kept[np.argsort(-plain_confidence, kind="stable")[:k]] = True

    def wrong(kept, idx=slice(None)): return int(((1 - correct[idx]) * kept[idx]).sum())
    def shown_sure_and_wrong(confidence, kept): return int(((1 - correct) * kept * (confidence >= 0.9)).sum())
    n = len(correct)
    def interval(statistic, resamples=2000):
        values = [statistic(rng.integers(0, n, n)) for _ in range(resamples)]
        return [round(float(np.percentile(values, 2.5)), 4), round(float(np.percentile(values, 97.5)), 4)]
    return {
        "images": n,
        "plain_answer_all_wrong": wrong(np.ones(n, bool)),
        "plain_own_threshold": {"coverage": round(plain_kept.mean(), 4), "answered": int(plain_kept.sum()), "wrong": wrong(plain_kept),
                                "accuracy_on_answered": round(correct[plain_kept].mean(), 4), "wrong_shown_at_90_or_more": shown_sure_and_wrong(plain_confidence, plain_kept)},
        "cassacheck": {"coverage": round(calibrated_kept.mean(), 4), "answered": k, "wrong": wrong(calibrated_kept),
                       "accuracy_on_answered": round(correct[calibrated_kept].mean(), 4), "wrong_shown_at_90_or_more": shown_sure_and_wrong(calibrated_confidence, calibrated_kept)},
        "plain_matched_coverage": {"answered": k, "wrong": wrong(matched_kept), "accuracy_on_answered": round(correct[matched_kept].mean(), 4)},
        "answered_sets_differ_on": int((matched_kept != calibrated_kept).sum()),
        "wrong_difference_own_thresholds_ci": interval(lambda idx: wrong(calibrated_kept, idx) - wrong(plain_kept, idx)),
        "wrong_difference_matched_ci": interval(lambda idx: wrong(calibrated_kept, idx) - wrong(matched_kept, idx)),
        "mean_confidence_shown_on_wrong_answers": {"plain": round(float(plain_confidence[(correct == 0) & plain_kept].mean()), 4),
                                                   "cassacheck": round(float(calibrated_confidence[(correct == 0) & calibrated_kept].mean()), 4)},
        "ece": {"plain": round(expected_calibration_error(plain, targets), 4), "cassacheck": round(expected_calibration_error(calibrated, targets), 4)},
        "ece_difference_ci": interval(lambda idx: expected_calibration_error(calibrated[idx], targets[idx]) - expected_calibration_error(plain[idx], targets[idx]), 500),
        "aurc": {"plain": round(area_under_risk_coverage(plain_confidence, correct), 4), "cassacheck": round(area_under_risk_coverage(calibrated_confidence, correct), 4)},
    }

results = {}
for backbone, (logits, targets) in test_logits.items():
    results[f"internal_test_{backbone}"] = compare(logits, targets, thresholds[backbone])
    print(f"=== internal test, {backbone} ===\n" + json.dumps(results[f"internal_test_{backbone}"], indent=1))

In [ ]:
# Cell 8: the same comparison on CCMT, from the logits saved by notebook 05 (labels from its manifest, brown spot excluded)
manifest = pd.read_csv(f"{RESULTS_DIRECTORY}/ccmt_manifest.csv")
for backbone, spec in frozen.items():
    logits = torch.load(f"{RESULTS_DIRECTORY}/ccmt_logits_{backbone}_seed{spec['seed']}.pt")
    assert len(logits) == len(manifest), "CCMT logits and manifest differ in length"
    targets = torch.tensor(manifest.label.values)
    results[f"ccmt_{backbone}"] = compare(logits, targets, thresholds[backbone])
    print(f"=== CCMT, {backbone} ===\n" + json.dumps(results[f"ccmt_{backbone}"], indent=1))

In [ ]:
# Cell 9: trivial baseline on the internal test: always answer CMD, the majority class
targets = test_logits["mobilenetv3_large"][1].numpy()
always_cmd = np.full(len(targets), 3)
results["internal_test_always_cmd"] = {"accuracy": round(float((always_cmd == targets).mean()), 4),
                                       "macro_f1": round(float(f1_score(targets, always_cmd, average="macro", zero_division=0)), 4)}
print(results["internal_test_always_cmd"])
json.dump(results, open(f"{RESULTS_DIRECTORY}/matched_coverage_comparison.json", "w"), indent=2)
print("saved", f"{RESULTS_DIRECTORY}/matched_coverage_comparison.json")